# GRPO Demo (GPU Version)

This tutorial demonstrates training the Gemma 2 2B-IT model on the GSM8K math reasoning benchmark using **Group Relative Policy Optimization (GRPO)**.

**This is the GPU version** using Hugging Face TRL library, which can run on NVIDIA GPUs.

## Requirements
- NVIDIA GPU with 24GB+ VRAM (RTX 3090/4090, A100, H100)
- CUDA 11.8+
- (Optional) Weights & Biases account

## Comparison with TPU Version

| Aspect | TPU Version | GPU Version |
|--------|-------------|-------------|
| Framework | Google Tunix + JAX | HuggingFace TRL + PyTorch |
| Hardware | TPU v5e-8 | NVIDIA GPU |
| Sharding | FSDP + TP | DeepSpeed / FSDP |
| Quantization | bfloat16 | 4-bit / 8-bit |


## Configuration


In [ ]:
# ============================================================
# USER CONFIGURATION - FILL IN BEFORE RUNNING
# ============================================================

# Hugging Face token (required for Gemma model access)
# Get from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <- Fill in your HF token here

# Wandb configuration (optional)
USE_WANDB = True
WANDB_API_KEY = ""  # <- Fill in your Wandb API key here (optional)

# GPU Memory optimization
USE_4BIT = True   # Use 4-bit quantization (recommended for <48GB VRAM)
USE_8BIT = False  # Use 8-bit quantization (alternative)

# ============================================================
# Apply configuration
# ============================================================
import os

os.environ["HF_TOKEN"] = HF_TOKEN

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    # Clear WANDB_DISABLED if previously set
    os.environ.pop("WANDB_DISABLED", None)
else:
    # Clear WANDB_API_KEY if previously set
    os.environ.pop("WANDB_API_KEY", None)
    os.environ.pop("WANDB_DISABLED", None)
    # Note: report_to="none" in GRPOConfig handles disabling wandb

print("Configuration applied!")
print(f"  4-bit quantization: {USE_4BIT}")
print(f"  8-bit quantization: {USE_8BIT}")
print(f"  Wandb Enabled: {USE_WANDB}")


Configuration applied!
  4-bit quantization: True
  8-bit quantization: False
  Wandb Enabled: True


## Imports


In [2]:
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login

# Login to Hugging Face
login(token=HF_TOKEN)

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


c:\Users\rog\.conda\envs\cv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\rog\.conda\envs\cv\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\rog\.conda\envs\cv\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rog\.conda\envs\cv\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you'v

PyTorch version: 2.9.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
VRAM: 34.2 GB


## Hyperparameters

These are configured to match the TPU version as closely as possible.


In [3]:
# ====== Model ======
MODEL_NAME = "google/gemma-2-2b-it"

# ====== LoRA ======
LORA_RANK = 64
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ====== GRPO ======
NUM_GENERATIONS = 4      # G in GRPO paper (same as TPU version)
BETA = 0.08              # KL penalty coefficient (same as TPU version)
EPSILON = 0.2            # Clipping epsilon (same as TPU version)

# ====== Generation ======
MAX_PROMPT_LENGTH = 256
MAX_COMPLETION_LENGTH = 512
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50

# ====== Training ======
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 1              # Reduce if OOM
GRADIENT_ACCUMULATION_STEPS = 8        # Effective batch size = 1 * 8 = 8
LEARNING_RATE = 3e-6
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 0.1
WEIGHT_DECAY = 0.1

# ====== Output ======
OUTPUT_DIR = "./grpo_output"
SAVE_STEPS = 500
LOGGING_STEPS = 10

print(f"Model: {MODEL_NAME}")
print(f"LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"GRPO: num_generations={NUM_GENERATIONS}, beta={BETA}, epsilon={EPSILON}")


Model: google/gemma-2-2b-it
LoRA rank: 64, alpha: 64
GRPO: num_generations=4, beta=0.08, epsilon=0.2


In [4]:
# Special tokens (same as TPU version)
REASONING_START = "<reasoning>"
REASONING_END = "</reasoning>"
ANSWER_START = "<answer>"
ANSWER_END = "</answer>"

SYSTEM_PROMPT = f"""You are given a problem. Think about the problem and \
provide your reasoning. Place it between {REASONING_START} and \
{REASONING_END}. Then, provide the final answer (i.e., just one numerical \
value) between {ANSWER_START} and {ANSWER_END}."""

# Gemma 2 chat template
def format_prompt(question):
    return f"""<start_of_turn>user
{SYSTEM_PROMPT}

{question}<end_of_turn>
<start_of_turn>model"""

# Regex patterns for reward functions
MATCH_FORMAT = re.compile(
    rf"^[\s]{{0,}}{REASONING_START}.+?{REASONING_END}.*?{ANSWER_START}(.+?){ANSWER_END}[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)
MATCH_NUMBERS = re.compile(rf"{ANSWER_START}.*?([\d\.]+)", flags=re.MULTILINE | re.DOTALL)


In [5]:
# Load GSM8K dataset
def extract_answer(text):
    """Extract numerical answer after ####"""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

def prepare_dataset(split="train", max_samples=None):
    dataset = load_dataset("openai/gsm8k", "main", split=split)
    
    def process(example):
        return {
            "prompt": format_prompt(example["question"]),
            "question": example["question"],
            "answer": extract_answer(example["answer"]),
        }
    
    dataset = dataset.map(process, remove_columns=dataset.column_names)
    dataset = dataset.filter(lambda x: x["answer"] is not None)
    
    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))
    
    return dataset

# Load datasets
train_dataset = prepare_dataset("train")
test_dataset = prepare_dataset("test", max_samples=200)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nSample prompt:\n{train_dataset[0]['prompt'][:200]}...")


Train samples: 7473
Test samples: 200

Sample prompt:
<start_of_turn>user
You are given a problem. Think about the problem and provide your reasoning. Place it between <reasoning> and </reasoning>. Then, provide the final answer (i.e., just one numerical...


## Load model with quantization


In [6]:
# Quantization config for memory efficiency
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
elif USE_8BIT:
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
else:
    quantization_config = None

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load model
print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
    attn_implementation="eager",  # For compatibility
)
model.config.use_cache = False

print(f"Model loaded! Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: google/gemma-2-2b-it...


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.11s/it]

Model loaded! Memory: 2.22 GB


## Apply LoRA


In [7]:
# LoRA configuration (same as TPU version)
peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

# Get trainable parameters count
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable, total = count_parameters(model)
print(f"Before LoRA - Trainable: {trainable:,} / Total: {total:,}")


Before LoRA - Trainable: 590,065,920 / Total: 1,602,203,904


## Reward functions

Same reward logic as TPU version:
- Format matching (exact and approximate)
- Answer correctness
- Number extraction


In [8]:
def reward_format_exact(completions, **kwargs):
    """Reward 3.0 if format matches exactly."""
    return [3.0 if MATCH_FORMAT.search(c) else 0.0 for c in completions]

def reward_format_approximate(completions, **kwargs):
    """Reward based on partial format matching."""
    scores = []
    for c in completions:
        score = 0.0
        score += 0.5 if c.count(REASONING_START) == 1 else -0.5
        score += 0.5 if c.count(REASONING_END) == 1 else -0.5
        score += 0.5 if c.count(ANSWER_START) == 1 else -0.5
        score += 0.5 if c.count(ANSWER_END) == 1 else -0.5
        scores.append(score)
    return scores

def reward_answer(completions, answer, **kwargs):
    """Reward based on answer correctness."""
    scores = []
    for c, true_ans in zip(completions, answer):
        match = MATCH_FORMAT.search(c)
        if not match:
            scores.append(0.0)
            continue
        guess = match.group(1)
        if guess == true_ans:
            scores.append(3.0)
        elif guess.strip() == true_ans.strip():
            scores.append(1.5)
        else:
            try:
                ratio = float(guess) / float(true_ans)
                if 0.9 <= ratio <= 1.1:
                    scores.append(0.5)
                elif 0.8 <= ratio <= 1.2:
                    scores.append(0.25)
                else:
                    scores.append(-1.0)
            except:
                scores.append(-0.5)
    return scores

def reward_numbers(completions, answer, **kwargs):
    """Extract number and check correctness."""
    scores = []
    for c, true_ans in zip(completions, answer):
        match = MATCH_NUMBERS.search(c)
        if not match:
            scores.append(0.0)
            continue
        try:
            guess = float(match.group(1).strip())
            true_val = float(true_ans.strip())
            scores.append(1.5 if guess == true_val else 0.0)
        except:
            scores.append(0.0)
    return scores

# Combined reward function for TRL
def reward_function(completions, prompts=None, **kwargs):
    """Combined reward function."""
    # Get answers from kwargs
    answer = kwargs.get("answer", [""] * len(completions))
    
    r1 = reward_format_exact(completions)
    r2 = reward_format_approximate(completions)
    r3 = reward_answer(completions, answer)
    r4 = reward_numbers(completions, answer)
    
    # Sum all rewards
    total = [a + b + c + d for a, b, c, d in zip(r1, r2, r3, r4)]
    return total

print("Reward functions defined!")


Reward functions defined!


In [9]:
# GRPO Training configuration
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    
    # Training
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    # Optimizer
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    weight_decay=WEIGHT_DECAY,
    optim="adamw_torch",
    
    # GRPO specific
    num_generations=NUM_GENERATIONS,
    beta=BETA,
    
    # Generation
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    
    # Logging & Saving
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=4,
    
    # Mixed precision
    bf16=True,
    
    # Disable wandb if not using
    report_to="wandb" if USE_WANDB else "none",
)

print("Training config created!")
print(f"  Effective batch size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")


Training config created!
  Effective batch size: 8


## Create trainer and train


In [10]:
# Create GRPO Trainer
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    peft_config=peft_config,
    reward_funcs=reward_function,
)

print("GRPO Trainer created!")
print(f"Training samples: {len(train_dataset)}")


GRPO Trainer created!
Training samples: 7473


In [ ]:
# Start training
print("Starting GRPO training...")
print("This may take a while. First step includes JIT compilation.")

trainer.train()

print("Training complete!")
trainer.save_model(f"{OUTPUT_DIR}/final")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Starting GRPO training...
This may take a while. First step includes JIT compilation.


wandb: Currently logged in as: cyiheng312 (cyiheng312-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,-0.036400
20,-0.037600
30,-0.031500
40,-0.075200
50,-0.013900
60,0.050800
70,0.027500
80,0.002600
90,0.002700
100,-0.021900


## Evaluation


In [ ]:
@torch.no_grad()
def evaluate_model(model, tokenizer, dataset, max_samples=100):
    """Evaluate model on test set."""
    model.eval()
    
    correct = 0
    partial_correct = 0
    format_correct = 0
    total = 0
    
    for i, sample in enumerate(dataset):
        if i >= max_samples:
            break
            
        prompt = sample["prompt"]
        true_answer = sample["answer"]
        
        # Generate
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=0.01,  # Greedy
            top_k=1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        # Check format
        if MATCH_FORMAT.search(response):
            format_correct += 1
        
        # Check answer
        match = MATCH_NUMBERS.search(response)
        if match:
            try:
                guess = float(match.group(1).strip())
                true_val = float(true_answer.strip())
                if guess == true_val:
                    correct += 1
                    partial_correct += 1
                elif 0.9 <= guess / true_val <= 1.1:
                    partial_correct += 1
            except:
                pass
        
        total += 1
        
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{max_samples}, Acc: {correct/total*100:.1f}%")
    
    return {
        "accuracy": correct / total * 100,
        "partial_accuracy": partial_correct / total * 100,
        "format_accuracy": format_correct / total * 100,
        "total": total,
    }

# Run evaluation
print("Evaluating trained model...")
results = evaluate_model(trainer.model, tokenizer, test_dataset, max_samples=100)

print(f"\n=== Evaluation Results ===")
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"Partial Accuracy: {results['partial_accuracy']:.2f}%")
print(f"Format Accuracy: {results['format_accuracy']:.2f}%")


## Test inference


In [ ]:
# Test with a sample question
test_question = "James has 30 teeth. His dentist drills 4 of them and caps 7 more teeth than he drills. What percentage of James' teeth does the dentist fix?"

prompt = format_prompt(test_question)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)
inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_k=50,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Question:", test_question)
print("\nModel Response:")
print(response)


## Summary

This GPU version uses:
- **Framework**: HuggingFace TRL + PyTorch (vs Google Tunix + JAX)
- **Quantization**: 4-bit NF4 (vs bfloat16)
- **Hardware**: NVIDIA GPU (vs TPU v5e-8)

### Key differences from TPU version:

| Aspect | TPU Version | GPU Version |
|--------|-------------|-------------|
| Framework | Google Tunix + JAX | HuggingFace TRL + PyTorch |
| Sharding | FSDP + TP on mesh | device_map="auto" |
| Memory | ~16GB per chip | 4-bit quantization |
| Batch processing | Grain | HuggingFace datasets |
| Checkpoint | Orbax | HuggingFace PEFT |

The training logic and reward functions are equivalent, so results should be comparable.
